# ¿Los perfiles sirven para recomendar?

Proyecto Integrador — Ciencia de Datos · UTN FRM · 2026 · Grupo 5K09-05

Hasta ahora juzgamos el modelo con la **silhouette**, que mide si los grupos quedan separados. Pero el
objetivo del proyecto es **recomendar películas**, y la silhouette no dice nada de eso. Este notebook
mide el modelo por lo que tiene que hacer.

**La idea, en una frase:** le escondemos a cada usuario algunas películas que ya calificó, le
recomendamos con distintos métodos sin dejarlos ver lo escondido, y contamos cuántas de las
recomendaciones son películas escondidas que le gustaron. Un método que recomienda bien "adivina" lo
que escondimos.

## Hoja de ruta

| Parte | Qué hacemos |
|---|---|
| **1** | Esconder ratings y rearmar todo sin ellos |
| **2** | Los métodos que comparamos |
| **3** | Medir: aciertos, precisión y variedad |
| **4** | Resultados |
| **5** | Un ejemplo: qué le recomienda cada método a un usuario |
| **6** | ¿Cuántas películas recomendar? |
| **7** | Conclusión |

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")

import altair as alt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

from modelado import COLUMNAS_DECADAS, SEMILLA, preparar_decadas, preparar_generos

alt.data_transformers.disable_max_rows()
pd.set_option("display.width", 160)
QUEMADO, CHARTREUSE, NAVY, ELECTRICO, GRIS, APAGADO = "#8b3a26", "#c4a22c", "#3e3f66", "#3b47c9", "#80818f", "#d4d4da"

ratings = pd.read_csv("ratings_selected_users.csv")              # los 3.503 usuarios modelados
peliculas = pd.read_csv("tmdb_movies_prepared.csv", usecols=["movieId", "titulo", "genres", "fecha_estreno"])
print(f"{len(ratings):,} ratings de {ratings['userId'].nunique():,} usuarios".replace(",", "."))

1.101.959 ratings de 3.503 usuarios


## Parte 1 — Esconder ratings sin hacer trampa

A cada usuario le escondemos **el 20 % de sus ratings, elegidos al azar**. Las películas escondidas que
calificó con **4 estrellas o más** son las que "debería" recomendarle un buen método: son las que
sabemos que le gustaron.

La trampa a evitar: el perfil de cada usuario (su afinidad por género, sus épocas) se calculó con
**todos** sus ratings, incluidos los que ahora escondemos. Si usáramos ese perfil, el modelo ya "sabría"
algo de lo escondido. Por eso **rearmamos la tabla silver sólo con los ratings que quedaron a la
vista**, con la misma cuenta que hace el DAG, y entrenamos el modelo de nuevo sobre esa tabla.

In [2]:
def rearmar(ratings, peliculas):
    """La tabla silver del DAG (una fila por usuario) a partir de una tabla de ratings."""
    base = ratings.groupby("userId")["rating"].agg(rating_promedio_general="mean", cantidad_valoraciones="size")
    g = peliculas.dropna(subset=["genres"]).assign(genero=lambda d: d["genres"].str.split("|")).explode("genero")
    por_genero = ratings.merge(g[["movieId", "genero"]], on="movieId").groupby(["userId", "genero"])["rating"].agg(
        ["mean", "size"]).unstack()
    anio = pd.to_datetime(peliculas["fecha_estreno"], errors="coerce").dt.year
    epoca = pd.cut(anio, [-np.inf, 1979, 1989, 1999, 2009, 2019, np.inf], labels=COLUMNAS_DECADAS).astype(str)
    re = ratings.merge(peliculas[["movieId"]].assign(epoca=epoca), on="movieId")
    props = pd.crosstab(re["userId"], re[re["epoca"] != "nan"]["epoca"], normalize="index").reindex(
        columns=COLUMNAS_DECADAS, fill_value=0)
    return (base.join(por_genero["mean"]).join(por_genero["size"].fillna(0).add_prefix("n_"))
            .join(props).fillna({c: 0 for c in COLUMNAS_DECADAS}))


# control: con todos los ratings, el rearmado da exactamente la tabla silver del DAG
silver = pd.read_csv("usuarios_generos.csv").set_index("userId")
control = rearmar(ratings, peliculas)
comunes = control.index
pd.Series({c: np.abs(control[c] - silver.loc[comunes, c]).max()
           for c in ["rating_promedio_general", "Drama", "n_Drama", "prop_1990s"]}, name="diferencia máxima")

rating_promedio_general    1.332268e-15
Drama                      8.881784e-16
n_Drama                    0.000000e+00
prop_1990s                 1.110223e-16
Name: diferencia máxima, dtype: float64

La diferencia máxima es 0: el rearmado reproduce la tabla del DAG. Ahora escondemos ratings y
preparamos todo lo que necesitan los métodos **sólo con lo que quedó a la vista**.

In [3]:
def preparar(semilla):
    rng = np.random.default_rng(semilla)
    oculto = rng.random(len(ratings)) < 0.2
    vista, escondida = ratings[~oculto], ratings[oculto]
    relevantes = escondida[escondida["rating"] >= 4].groupby("userId")["movieId"].apply(set)

    # el modelo de la Entrega 3, entrenado de nuevo sólo con lo que quedó a la vista
    D = rearmar(vista, peliculas)
    zg, zd = preparar_generos().fit_transform(D), preparar_decadas().fit_transform(D)
    perfil = (KMeans(2, random_state=SEMILLA, n_init=10).fit_predict(zg) * 2
              + KMeans(2, random_state=SEMILLA, n_init=10).fit_predict(zd))
    usuarios = D.index.to_numpy()
    pos = {u: i for i, u in enumerate(usuarios)}

    # matrices usuario × película (sólo películas con datos de TMDB, como en la app)
    candidatas = np.array(sorted(set(vista["movieId"]) & set(peliculas["movieId"])))
    col = {m: j for j, m in enumerate(candidatas)}
    v = vista[vista["movieId"].isin(col)]
    f, c = v["userId"].map(pos).to_numpy(), v["movieId"].map(col).to_numpy()
    forma = (len(usuarios), len(candidatas))
    M = sparse.csr_matrix((v["rating"].to_numpy(), (f, c)), shape=forma)                        # rating
    C = sparse.csr_matrix((np.ones(len(v)), (f, c)), shape=forma)                                # la vio
    B = sparse.csr_matrix(((v["rating"].to_numpy() >= 4).astype(float), (f, c)), shape=forma)     # le gustó

    # cuánto se parecen dos usuarios: con el modelo (como la app) o por las películas que comparten
    dist_modelo = (euclidean_distances(zg) / np.sqrt(zg.shape[1]) + euclidean_distances(zd) / np.sqrt(zd.shape[1])) / 2
    centrada = M.copy()
    medias = np.asarray(M.sum(1)).ravel() / np.maximum(np.asarray(C.sum(1)).ravel(), 1)
    centrada.data = centrada.data - np.repeat(medias, np.diff(M.indptr))
    sim_peliculas = cosine_similarity(centrada)
    return dict(relevantes=relevantes, perfil=perfil, usuarios=usuarios, pos=pos, candidatas=candidatas,
                M=M, C=C, B=B, vistas=C.tolil().rows, dist_modelo=dist_modelo, sim_peliculas=sim_peliculas,
                rng=rng)

## Parte 2 — Los métodos que comparamos

Todos recomiendan **10 películas** que el usuario no calificó (entre las que quedaron a la vista), y
todos eligen entre las 9.928 películas con datos de TMDB, como la app.

| Método | Cómo elige | Para qué está |
|---|---|---|
| **Al azar** | 10 películas cualesquiera | El piso: cualquier método tiene que ganarle por mucho. |
| **Más populares** | Las que más usuarios calificaron con 4 o más | La base que hay que superar: es lo que haría cualquiera sin modelo. |
| **Más populares de su perfil** | Lo mismo, pero sólo entre los usuarios de su perfil (de los 4) | Usa los grupos directamente. |
| **Vecinos del modelo · regla de la app** | Los 100 usuarios más cercanos en el espacio del modelo; películas que vieron al menos 2 de ellos con promedio de 3,8 o más, ordenadas por promedio | Es **exactamente lo que hace hoy la app**. |
| **Vecinos del modelo · votos** | Los mismos 100 vecinos; cada película suma un voto por cada vecino que la calificó con 4 o más, pesado por cercanía | Mismo modelo, otra regla para elegir. |
| **Vecinos por películas** · regla de la app / votos | Los 100 usuarios que calificaron parecido **las mismas películas** (similitud coseno), con cada una de las dos reglas | La forma clásica de recomendar (filtrado colaborativo), sin perfiles. Sirve de techo. |

Así se pueden separar dos preguntas: **¿el espacio del modelo encuentra buenos vecinos?** (comparando
con los vecinos por películas) y **¿la regla de la app elige bien entre lo que ven los vecinos?**
(comparando las dos reglas con los mismos vecinos).

In [4]:
N = 10
METODOS = ["Al azar", "Más populares", "Más populares de su perfil",
           "Vecinos del modelo · regla de la app", "Vecinos del modelo · votos",
           "Vecinos por películas · regla de la app", "Vecinos por películas · votos"]


def recomendar(P, metodo, i):
    cand, vistas = P["candidatas"], P["vistas"][i]

    def mejores(puntaje):
        puntaje = puntaje.astype(float).copy()
        puntaje[vistas] = -np.inf
        orden = np.argpartition(-puntaje, N)[:N]
        orden = orden[np.argsort(-puntaje[orden])]
        return [cand[j] for j in orden if np.isfinite(puntaje[j])]

    if metodo == "Al azar":
        return list(P["rng"].choice(np.delete(cand, vistas), N, replace=False))
    if metodo == "Más populares":
        return mejores(np.asarray(P["B"].sum(0)).ravel())
    if metodo == "Más populares de su perfil":
        return mejores(np.asarray(P["B"][P["perfil"] == P["perfil"][i]].sum(0)).ravel())

    # los 100 vecinos y su peso (más cerca o más parecido, más peso)
    if metodo.startswith("Vecinos del modelo"):
        d = P["dist_modelo"][i].copy(); d[i] = np.inf
        idx = np.argpartition(d, 100)[:100]; pesos = 1 / (d[idx] + 0.001)
    else:
        s = P["sim_peliculas"][i].copy(); s[i] = -np.inf
        idx = np.argpartition(-s, 100)[:100]; pesos = np.clip(s[idx], 0, None)
    fila = np.zeros(len(idx), int)
    w = sparse.csr_matrix((pesos, (fila, idx)), shape=(1, len(P["usuarios"])))
    if metodo.endswith("votos"):
        return mejores((w @ P["B"]).toarray().ravel())
    uno = sparse.csr_matrix((np.ones(len(idx)), (fila, idx)), shape=(1, len(P["usuarios"])))
    cuantos = (uno @ P["C"]).toarray().ravel()
    promedio = (uno @ P["M"]).toarray().ravel() / np.maximum(cuantos, 1)
    ponderado = (w @ P["M"]).toarray().ravel() / np.maximum((w @ P["C"]).toarray().ravel(), 1e-9)
    return mejores(np.where((cuantos >= 2) & (promedio >= 3.8), ponderado + cuantos * 1e-6, -np.inf))

## Parte 3 — Cómo medimos

Para cada usuario, con sus 10 recomendaciones y sus películas escondidas que le gustaron:

- **Acierta:** si al menos una de las 10 es una película escondida que le gustó.
- **Precisión@10:** qué parte de las 10 son películas escondidas que le gustaron. 0,20 = 2 de 10.
- **Recall@10:** qué parte de las películas escondidas que le gustaron aparece entre las 10.
- **Variedad:** cuántas películas distintas recomienda el método sumando a todos los usuarios. Un
  método que le recomienda lo mismo a todos tiene poca variedad aunque acierte.

Es una medida **exigente**: si el método recomienda una película excelente que el usuario nunca vio,
cuenta como error, porque no sabemos si le gustaría. Por eso los números absolutos son bajos; lo que
importa es la **comparación** entre métodos.

Repetimos todo con **tres maneras distintas de esconder** (semillas 42, 7 y 123) para ver si los
resultados dependen de qué ratings escondimos.

In [5]:
filas, recomendadas = [], {}
for semilla in [42, 7, 123]:
    P = preparar(semilla)
    for u, rel in P["relevantes"].items():
        rel = rel & set(P["candidatas"])
        if not rel:
            continue
        i = P["pos"][u]
        for m in METODOS:
            rec = recomendar(P, m, i)
            recomendadas.setdefault((semilla, m), set()).update(rec)
            aciertos = len(set(rec) & rel)
            filas.append({"semilla": semilla, "metodo": m, "acierta": aciertos > 0,
                          "precision": aciertos / N, "recall": aciertos / len(rel)})
res = pd.DataFrame(filas)
por_semilla = res.groupby(["metodo", "semilla"])[["acierta", "precision", "recall"]].mean()
por_semilla["variedad"] = [len(recomendadas[(s, m)]) for m, s in por_semilla.index]
resumen = por_semilla.groupby("metodo").agg(["mean", "min", "max"]).reindex(METODOS)
print(f"Usuarios evaluados por semilla: {res.groupby('semilla').size().iloc[0] // len(METODOS):,}".replace(",", "."))
resumen[[("acierta", "mean"), ("precision", "mean"), ("precision", "min"), ("precision", "max"),
         ("recall", "mean"), ("variedad", "mean")]].round(3)

Usuarios evaluados por semilla: 3.499


acierta precision               recall  variedad
                                           mean      mean    min    max   mean      mean
metodo                                                                                  
Al azar                                   0.030     0.003  0.003  0.003  0.001  9620.667
Más populares                             0.772     0.201  0.200  0.201  0.075    56.000
Más populares de su perfil                0.802     0.224  0.223  0.225  0.083   109.667
Vecinos del modelo · regla de la app      0.040     0.004  0.004  0.004  0.001  3183.667
Vecinos del modelo · votos                0.848     0.266  0.264  0.268  0.102   285.333
Vecinos por películas · regla de la app   0.065     0.007  0.007  0.008  0.002  3326.667
Vecinos por películas · votos             0.875     0.290  0.289  0.291  0.111   261.000

## Parte 4 — Resultados

In [6]:
datos = resumen["precision"].reset_index()
datos["familia"] = np.select([datos["metodo"].str.contains("regla de la app"), datos["metodo"].str.contains("votos"),
                              datos["metodo"].isin(["Al azar"])],
                             ["Regla de la app", "Votos", "Piso"], "Popularidad")
datos["t"] = datos["mean"].map(lambda v: f"{v:.3f}".replace(".", ","))
y = alt.Y("metodo:N", sort=METODOS, title=None, axis=alt.Axis(labelLimit=0))
escala = alt.Scale(domain=["Piso", "Popularidad", "Regla de la app", "Votos"], range=[APAGADO, GRIS, QUEMADO, ELECTRICO])
barras = alt.Chart(datos).mark_bar(size=20, cornerRadiusEnd=4).encode(
    x=alt.X("mean:Q", title="Precisión@10 (qué parte de las 10 recomendaciones le gustó)", scale=alt.Scale(domain=[0, 0.32])),
    y=y, color=alt.Color("familia:N", scale=escala, title=None, legend=alt.Legend(orient="top")),
    tooltip=["metodo", alt.Tooltip("mean:Q", title="promedio", format=".3f"),
             alt.Tooltip("min:Q", title="mínimo", format=".3f"), alt.Tooltip("max:Q", title="máximo", format=".3f")])
rango = alt.Chart(datos).mark_rule(color="#26273d", strokeWidth=1.5).encode(x="min:Q", x2="max:Q", y=y)
texto = alt.Chart(datos).mark_text(align="left", dx=6, color="#26273d", fontSize=12).encode(x="max:Q", y=y, text="t:N")
(barras + rango + texto).properties(width=520, height=260, title=alt.Title(
    "La regla de la app casi no acierta; con votos, el mismo modelo le gana a las más populares",
    subtitle="Promedio de tres maneras de esconder ratings; la rayita negra es el rango entre las tres.",
    anchor="start", fontSize=15)).configure_view(stroke=None)

alt.LayerChart(...)

**Lo que muestra el gráfico:**

1. **La regla de la app no sirve para recomendar.** Acierta apenas un poco más que el azar. Ordena por el
   **promedio** de los vecinos y acepta películas que vieron sólo 2 de ellos: así sube al primer lugar
   cualquier película rara que dos vecinos calificaron con 5. Es la misma trampa de los pocos ratings de
   la sección 2 de la app, ahora del lado de las películas.
2. **Con votos, el modelo funciona.** Los mismos vecinos del modelo, contando cuántos calificaron bien
   cada película, superan a las más populares y a las más populares del perfil. O sea: **el espacio
   del modelo sí encuentra usuarios parecidos**.
3. **Los vecinos por películas son un poco mejores todavía.** Comparar a los usuarios por las películas
   que comparten le gana al modelo por poco. Es esperable: el modelo resume a cada usuario en géneros y
   épocas y pierde detalle.
4. **Hay un costo en variedad.** Los métodos de popularidad recomiendan casi lo mismo a todos; los de
   vecinos reparten las recomendaciones entre muchas más películas.

In [7]:
var = resumen["variedad"]["mean"].reset_index().merge(resumen["precision"]["mean"].reset_index(), on="metodo",
                                                     suffixes=("_variedad", "_precision"))
var.columns = ["metodo", "variedad", "precision"]
var["familia"] = datos.set_index("metodo").loc[var["metodo"], "familia"].to_numpy()
alt.Chart(var).mark_point(filled=True, size=140).encode(
    x=alt.X("variedad:Q", title="Películas distintas recomendadas (escala log)", scale=alt.Scale(type="log")),
    y=alt.Y("precision:Q", title="Precisión@10"),
    color=alt.Color("familia:N", scale=escala, title=None, legend=alt.Legend(orient="top")),
    tooltip=["metodo", alt.Tooltip("variedad:Q", format=",.0f"), alt.Tooltip("precision:Q", format=".3f")]
).properties(width=520, height=300, title=alt.Title(
    "Los vecinos con votos aciertan más y además recomiendan más variado",
    subtitle="Arriba a la derecha es mejor. Pasá el mouse por un punto para ver el método.",
    anchor="start", fontSize=15)).configure_view(stroke=None)

alt.Chart(...)

## Parte 5 — Un ejemplo

Para que se vea qué significa, un usuario al que la regla de votos le acierta varias: lo que le
recomiendan la regla de la app y la de votos con los mismos vecinos. No es un caso típico (en promedio,
votos acierta 2 o 3 de 10); sirve para ver qué tipo de película elige cada regla.

In [8]:
P = preparar(42)
titulo = peliculas.set_index("movieId")["titulo"]
# un usuario donde la regla de votos acierta al menos 3 (no es el caso típico: la precisión media es 0,27)
for u, r in P["relevantes"].items():
    rel = r & set(P["candidatas"])
    i = P["pos"][u]
    if len(rel) >= 8 and len(set(recomendar(P, "Vecinos del modelo · votos", i)) & rel) >= 3:
        break
ejemplo = pd.DataFrame({
    "Regla de la app": [titulo.get(m, m) + ("  ✔" if m in rel else "")
                        for m in recomendar(P, "Vecinos del modelo · regla de la app", i)],
    "Votos": [titulo.get(m, m) + ("  ✔" if m in rel else "") for m in recomendar(P, "Vecinos del modelo · votos", i)],
})
print(f"Usuario {u}: le escondimos {len(rel)} películas que le gustaron. ✔ = acierto.")
ejemplo

Usuario 323: le escondimos 12 películas que le gustaron. ✔ = acierto.


,Regla de la app,Votos
0,On the Waterfront,The Shawshank Redemption
1,The Celebration,The Godfather
2,The Castle,Schindler's List ✔
3,Dancing at Lughnasa,Forrest Gump ✔
4,Harvey,Star Wars ✔
5,A Man for All Seasons,As Good as It Gets
6,"Fast, Cheap & Out of Control",The Fugitive ✔
7,Mutiny on the Bounty,Groundhog Day
8,Paths of Glory,Se7en
9,Local Hero,American Beauty


## Parte 6 — ¿Cuántas películas recomendar?

Hasta acá cada método recomendaba 10. Pero no todas las recomendaciones valen lo mismo: la primera es la
película que más vecinos votaron, la décima tiene menos apoyo. ¿Dónde conviene cortar?

Probamos dos ideas:

- **Corte por salto:** si hubiera una caída brusca entre la película 5 y la 6, quedarse con 5.
- **Corte por calidad:** recomendar mientras cada película tenga una chance razonable de gustar.

Para las dos hace falta lo mismo: la **precisión de cada posición**, o sea, de las películas que quedaron
en el puesto 1, 2, 3…, qué parte eran películas escondidas que le gustaban al usuario. La calculamos para
las 20 primeras, con votos de los vecinos del modelo y, para comparar, con las más populares.

In [9]:
filas = []
for semilla in [42, 7, 123]:
    P = preparar(semilla)
    populares = np.asarray(P["B"].sum(0)).ravel().astype(float)
    for u, rel in P["relevantes"].items():
        rel = rel & set(P["candidatas"])
        if not rel:
            continue
        i = P["pos"][u]
        d = P["dist_modelo"][i].copy(); d[i] = np.inf
        idx = np.argpartition(d, 100)[:100]
        w = sparse.csr_matrix((1 / (d[idx] + 0.001), (np.zeros(100, int), idx)), shape=(1, len(P["usuarios"])))
        for metodo, puntaje in [("Votos de los vecinos del modelo", (w @ P["B"]).toarray().ravel()),
                                ("Más populares", populares.copy())]:
            puntaje[P["vistas"][i]] = -np.inf
            for posicion, j in enumerate(np.argsort(-puntaje)[:20], start=1):
                filas.append({"semilla": semilla, "metodo": metodo, "posicion": posicion,
                              "acierto": P["candidatas"][j] in rel})
por_posicion = pd.DataFrame(filas).groupby(["metodo", "posicion"])["acierto"].mean().reset_index(name="precision")
por_posicion.pivot(index="posicion", columns="metodo", values="precision").round(3).T

posicion,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
metodo,,,,,,,,,,,,,,,,,,,,
Más populares,0.297,0.269,0.236,0.209,0.196,0.176,0.164,0.161,0.153,0.145,0.138,0.131,0.136,0.129,0.128,0.123,0.116,0.119,0.112,0.109
Votos de los vecinos del modelo,0.402,0.341,0.304,0.276,0.253,0.243,0.228,0.215,0.207,0.195,0.191,0.181,0.180,0.166,0.168,0.159,0.162,0.156,0.141,0.142


In [10]:
UMBRAL = 0.20   # una de cada cinco
votos = por_posicion[por_posicion["metodo"] == "Votos de los vecinos del modelo"].set_index("posicion")["precision"]
N_ELEGIDO = int(votos[votos >= UMBRAL].index.max())
print(f"Con votos, cada una de las primeras {N_ELEGIDO} películas acierta al menos {UMBRAL:.0%} de las veces; "
      f"la {N_ELEGIDO + 1}.ª, {votos[N_ELEGIDO + 1]:.1%}.")

x = alt.X("posicion:Q", title="Posición de la recomendación (1 = la primera)", scale=alt.Scale(domain=[1, 20]),
          axis=alt.Axis(values=list(range(1, 21)), format="d", grid=False))
y = alt.Y("precision:Q", title="Precisión de esa posición", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0, 0.45]))
lineas = alt.Chart(por_posicion).mark_line(strokeWidth=2, point=alt.OverlayMarkDef(size=40)).encode(
    x=x, y=y, color=alt.Color("metodo:N", title=None, legend=alt.Legend(orient="top", labelLimit=0),
                              scale=alt.Scale(domain=["Votos de los vecinos del modelo", "Más populares"],
                                              range=[ELECTRICO, GRIS])),
    tooltip=["metodo", "posicion", alt.Tooltip("precision:Q", format=".1%")])
umbral = alt.Chart(pd.DataFrame({"y": [UMBRAL]})).mark_rule(color=QUEMADO, strokeDash=[4, 3]).encode(y="y:Q")
texto_u = alt.Chart(pd.DataFrame({"x": [20], "y": [UMBRAL], "t": ["1 de cada 5"]})).mark_text(
    align="right", dy=-7, color=QUEMADO, fontSize=11).encode(x="x:Q", y="y:Q", text="t:N")
corte = alt.Chart(pd.DataFrame({"x": [N_ELEGIDO + 0.5]})).mark_rule(color="#26273d").encode(x="x:Q")
texto_c = alt.Chart(pd.DataFrame({"x": [N_ELEGIDO + 0.5], "t": [f"cortamos en {N_ELEGIDO}"]})).mark_text(
    align="left", dx=5, baseline="top", color="#26273d", fontSize=11).encode(x="x:Q", y=alt.value(4), text="t:N")
(lineas + umbral + texto_u + corte + texto_c).properties(width=560, height=300, title=alt.Title(
    f"La precisión baja de a poco: no hay salto, pero hasta la {N_ELEGIDO}.ª cada una acierta 1 de cada 5",
    subtitle="Promedio de las tres maneras de esconder ratings. Votos le gana a las más populares en todas las posiciones.",
    anchor="start", fontSize=15)).configure_view(stroke=None)

Con votos, cada una de las primeras 9 películas acierta al menos 20% de las veces; la 10.ª, 19.5%.


alt.LayerChart(...)

**Lo que muestra:**

- **No hay salto.** La precisión baja de forma pareja de una posición a la siguiente, así que el corte
  por salto no tiene dónde cortar: la diferencia entre la 5.ª y la 6.ª es la misma que entre la 6.ª y la
  7.ª.
- **Sí hay un criterio de calidad claro.** La primera recomendación acierta 4 de cada 10 veces. Hasta la
  posición marcada, cada película acierta al menos 1 de cada 5 veces; desde la siguiente, menos. Ese es el
  corte que usa la app.
- **Votos le gana a las más populares en todas las posiciones**, no sólo en el promedio de las 10.

## Parte 7 — Conclusión

| Pregunta | Respuesta |
|---|---|
| ¿El modelo sirve para recomendar? | **Sí**, si se lo usa bien: los vecinos del modelo con votos superan a recomendar las más populares, también a las populares de cada perfil. |
| ¿Y la app, como está hoy? | **No.** La regla que usa (promedio de los vecinos, con sólo 2 vecinos) acierta casi como el azar. |
| ¿Por qué? | Porque premia películas raras que dos vecinos calificaron alto: la trampa de los pocos ratings, del lado de las películas. |
| ¿Se puede recomendar mejor sin los perfiles? | Un poco: comparar usuarios por las películas que comparten acierta algo más que el espacio del modelo. |

**Qué haríamos:**

1. **Cambiar la regla de la app por votos.** Es un cambio chico en la función `recomendar` y es la
   mejora más grande que encontramos.
2. **Recomendar 9 películas**: cada una de las primeras 9 acierta al menos 1 de cada 5 veces; la 10.ª, apenas menos (Parte 6).
3. Usar esta evaluación como **la métrica del proyecto**: cualquier cambio al modelo (otras variables,
   como el director; otra representación, como SVD o NMF) se compara con estos números.